### Combine datasets from previous research 
This notebook is about downloading data

In [97]:
# Install packages
# %pip install pandas
# %pip install fuzzywuzzy
# %pip install pycountry

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 32.4 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [98]:
import pandas as pd
import re
from fuzzywuzzy import fuzz
import pycountry

In [3]:
#Load data from Sachdeva et al paper
# sachdeva_data = pd.read_csv('data/net_zero_cities_full_dataset.csv')

# sachdeva_data['plan_name'] = sachdeva_data['text'].str[:100]
# sachdeva_data_clip = sachdeva_data['name', 'iso', 'plan_name']

# sachdeva_data_clip.to_csv('sachdeva_plans.csv')


            name  iso entity_type         country                     region  \
0  Aberdeenshire  GBR      Region  United Kingdom                     Europe   
1     Acquappesa  ITA        City           Italy                     Europe   
2     Acquappesa  ITA        City           Italy                     Europe   
3       Adelaide  AUS        City       Australia  East Asia and the Pacific   
4       Adelaide  AUS        City       Australia  East Asia and the Pacific   

         lat         lng  population  population_year  baseline_emissions  \
0  57.166667   -2.666667    243510.0              NaN                 NaN   
1  39.500000   15.950000      1882.0           2018.0        8.575571e+03   
2  39.500000   15.950000      1882.0           2018.0        8.575571e+03   
3 -34.928889  138.601111     23916.0           2017.0        1.148108e+06   
4 -34.928889  138.601111     23916.0           2017.0        1.148108e+06   

   ...                            raw_net_zero_commitmen

In [149]:
#Get city population data from UN World Urban Projection data
#Download Link: https://population.un.org/wup/Download/Files/WUP2018-F12-Cities_Over_300K.xls
#Latest Download Date: Feb 15, 2024
city_pop_data = pd.read_excel('data/WUP2018-F22-Cities_Over_300K_Annual.xls', skiprows=16)
#Download link: https://population.un.org/wup/Download/Files/WUP2018-F13-Capital_Cities.xls
#Latest Download Date: Feb 15, 2024
cap_city_data = pd.read_excel('data/WUP2018-F13-Capital_Cities.xls', skiprows=16)


#Using 2018 data becaause that was the last update and data for capital cities is from then
# Next update is in 2025
cities_1mill = city_pop_data[city_pop_data[2018] > 1000]

#get capital cities that have populations < 1million
cap_cities = cap_city_data[cap_city_data['Population (thousands)'] <= 1000]
# print(cities_1mill.head)
print(cap_cities.head())

   Index      Country or area  Country code      Capital City  Note  \
1      2              Albania             8   Tiranë (Tirana)   NaN   
3      4       American Samoa            16         Pago Pago   NaN   
4      5              Andorra            20  Andorra la Vella   NaN   
6      7             Anguilla           660        The Valley   NaN   
7      8  Antigua and Barbuda            28        St. John's   NaN   

  Capital Type  City code  Latitude  Longitude  Population (thousands)  \
1      Capital      20005  41.32750   19.81889                 475.577   
3      Capital      20047 -14.27806 -170.70250                  48.526   
4      Capital      20048  42.50779    1.52109                  22.614   
6      Capital      22407  18.21704  -63.05783                   1.402   
7      Capital      20052  17.11717  -61.84573                  20.764   

   Unnamed: 10  Unnamed: 11  
1          NaN          NaN  
3          NaN          NaN  
4          NaN          NaN  
6       

In [160]:
#combine datasets of cities over 1mill with capital cities
city_list = pd.concat([cities_1mill, cap_cities], join='outer', ignore_index=True)
city_list['population'] = city_list['Population (thousands)'].combine_first(city_list[2018])
city_list['city'] = city_list['Urban Agglomeration'].combine_first(city_list['Capital City'])

city_list = city_list.rename(columns={'Country code':'country_code', 'City code': 'city_code', 'Country or area': 'country'} )
city_list = city_list[['country', 'country_code', 'city', 'city_code', 'Latitude', 'Longitude', 'population']]

city_list['city_code'] = city_list['city_code'].astype(str)

print(city_list.head)


<bound method NDFrame.head of                           country  country_code                   city  \
0                     Afghanistan           NaN                  Kabul   
1                         Algeria           NaN  El Djazaïr  (Algiers)   
2                          Angola           NaN                 Luanda   
3                       Argentina           NaN           Buenos Aires   
4                       Argentina           NaN                Córdoba   
..                            ...           ...                    ...   
666   United Republic of Tanzania         834.0                 Dodoma   
667  United States Virgin Islands         850.0       Charlotte Amalie   
668                       Vanuatu         548.0              Port Vila   
669     Wallis and Futuna Islands         876.0               Matu-Utu   
670                Western Sahara         732.0               El Aaiún   

    city_code   Latitude   Longitude  population  
0         nan  34.528887   69.

In [85]:
#write list to file
city_list.to_csv('output/capital_and_1mill_pop_cities.csv')

In [156]:
#Get and clean CDP 2023 Climate Actions Plan data
#Download link: https://data.cdp.net/Mitigation-Actions/2023-Cities-Climate-Action-Plans/t3a4-f7sg
#Latest Download Date: Feb 23, 2024

cdp2023_plans = pd.read_csv('data/2023_Cities_Climate_Action_Plans_20240223.csv')

cdp2023_plans_cleaned = cdp2023_plans[['Organization Number', 'Organization Name', 'City', 'Country/Area', 'Attachment/ link and name of plan', 'Year of formal approval of plan', 'City Location' ]]
cdp2023_plans_cleaned = cdp2023_plans_cleaned.rename(columns={'Organization Number': 'CDP_org_code', 'Year of formal approval of plan': 'Plan Year', 'Country/Area' : 'Country'})

#Data cleaning
#Get city name
def fill_city(row):
    if pd.notna(row['City']):
        return row['City']
    else:
        return str(row['Organization Name']).replace('City of ', '').replace('Municipality of ', '').replace('Municipality', '').replace('Municipalidad de ', '')

cdp2023_plans_cleaned['City'] = cdp2023_plans_cleaned.apply(fill_city, axis=1)

#Get link or PDF name
def extract_link(row):
    # Define a regular expression to find links
    link_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    links = re.findall(link_pattern, str(row['Attachment/ link and name of plan']))
    
    return links[0] if links else None

def extract_pdf_name(row):
    # If no links are found, try to find a PDF name using another pattern
    pdf_name_pattern = re.compile(r'\b\w+\.pdf\b', flags=re.IGNORECASE)
    pdf_names = pdf_name_pattern.findall(str(row['Attachment/ link and name of plan']))

    return pdf_names[0] if pdf_names else None

cdp2023_plans_cleaned['Plan Link'] =  cdp2023_plans_cleaned.apply(extract_link, axis=1)
cdp2023_plans_cleaned['Plan Name'] =  cdp2023_plans_cleaned.apply(extract_pdf_name, axis=1)
cdp2023_plans_cleaned['Plan Year']

print(cdp2023_plans_cleaned)

#write cleaned up list to file
# existing_plans.to_csv('output/existing-plans-cleaned.csv')

      CDP_org_code                      Organization Name  \
0           847232                           Tres Arroyos   
1            54478               Municipality of Nijmegen   
2            54389               Taichung City Government   
3            54308  Coimbatore City Municipal Corporation   
4            54386                 Tainan City Government   
...            ...                                    ...   
1198         37261               City of Pietermaritzburg   
1199         10495                  City of Las Vegas, NV   
1200         36004                        City of Abidjan   
1201         35894                  Ville de Montreal, QC   
1202        848125                               Oreamuno   

                                       City                   Country  \
0                              Tres Arroyos                 Argentina   
1                                  Nijmegen               Netherlands   
2                                  Taichung     

In [152]:
#Merge existing list of plans with CDP data
#Get existing list of plans
manual_existing_plans = pd.read_csv('data/Data - List of Existing Cities_Plans.csv')
existing_plans = pd.concat([manual_existing_plans, cdp2023_plans_cleaned], join='outer', ignore_index=True)

# Function to get ISO code for a given country name
def get_iso_code(country_name):
    try:
        # Use pycountry to get ISO code
        country = pycountry.countries.search_fuzzy(str(country_name))[0]
        return country.alpha_3
    except LookupError:
        # Return None if the country name is not found
        return None
    
def get_languages(country_code):
    try:
        country = pycountry.countries.get(alpha_3=country_code)
        if country:
            return [lang.name for lang in country.languages]
        else:
            return None
    except Exception as e:
        print(f"Error: {e}")
        return None
    

# existing_plans['Languages'] = existing_plans['ISO_Code'].apply(get_languages)

# print(existing_plans.head())

SubdivisionHierarchy(code='TH-55', country_code='TH', name='Nan', parent_code=None, type='Province')
SubdivisionHierarchy(code='US-AK', country_code='US', name='Alaska', parent_code=None, type='State')
SubdivisionHierarchy(code='US-AL', country_code='US', name='Alabama', parent_code=None, type='State')
SubdivisionHierarchy(code='US-AR', country_code='US', name='Arkansas', parent_code=None, type='State')
SubdivisionHierarchy(code='US-AS', country_code='US', name='American Samoa', parent_code=None, type='Outlying area')
SubdivisionHierarchy(code='US-AZ', country_code='US', name='Arizona', parent_code=None, type='State')
SubdivisionHierarchy(code='US-CA', country_code='US', name='California', parent_code=None, type='State')
SubdivisionHierarchy(code='US-CO', country_code='US', name='Colorado', parent_code=None, type='State')
SubdivisionHierarchy(code='US-CT', country_code='US', name='Connecticut', parent_code=None, type='State')
SubdivisionHierarchy(code='US-DC', country_code='US', name='

In [161]:
existing_plans['ISO_Code'] = existing_plans['Country'].apply(get_iso_code)
city_list['ISO_Code'] = city_list['country'].apply(get_iso_code)


SubdivisionHierarchy(code='TH-55', country_code='TH', name='Nan', parent_code=None, type='Province')
SubdivisionHierarchy(code='US-AK', country_code='US', name='Alaska', parent_code=None, type='State')
SubdivisionHierarchy(code='US-AL', country_code='US', name='Alabama', parent_code=None, type='State')
SubdivisionHierarchy(code='US-AR', country_code='US', name='Arkansas', parent_code=None, type='State')
SubdivisionHierarchy(code='US-AS', country_code='US', name='American Samoa', parent_code=None, type='Outlying area')
SubdivisionHierarchy(code='US-AZ', country_code='US', name='Arizona', parent_code=None, type='State')
SubdivisionHierarchy(code='US-CA', country_code='US', name='California', parent_code=None, type='State')
SubdivisionHierarchy(code='US-CO', country_code='US', name='Colorado', parent_code=None, type='State')
SubdivisionHierarchy(code='US-CT', country_code='US', name='Connecticut', parent_code=None, type='State')
SubdivisionHierarchy(code='US-DC', country_code='US', name='

In [167]:
#Merge existing plans with list of desired cities

#TODO: try with small list of cities to see what problem is occuring

def find_matching_city(row, city_list_group):
    # city_list_group_lower = city_list_group['city'].str.lower()
    # row_city_lower = row['City'].lower()

    # Check if the city names match or if one is contained in the other
    matches = city_list_group.str.apply(lambda city: city.lower() in row['City'].lower() or row['City'].lower()in city.lower())
    matching_city_codes = city_list_group.loc[matches, 'city_code'].tolist()

    if matching_city_codes:
        return ', '.join(matching_city_codes)
    else:
        return None


existing_plans['matched_cities'] = existing_plans.apply(
    lambda row: find_matching_city(row, city_list.groupby('ISO_Code').get_group(row['ISO_Code'])) 
    if pd.notna(row['ISO_Code']) and row['ISO_Code'] in city_list['ISO_Code'].values
    else None,
    axis=1
)

# Display the result
# print(existing_plans)

# merged_dataset = existing_plans.merge(city_list, left_on='matched_cities', right_on='city_code', how='left')
# print(merged_dataset)

                  City                   Country  \
0            Amsterdam               Netherlands   
1            Amsterdam               Netherlands   
2            Amsterdam               Netherlands   
3            Amsterdam               Netherlands   
4            Amsterdam               Netherlands   
...                ...                       ...   
1639  Pietermaritzburg              South Africa   
1640         Las Vegas  United States of America   
1641           Abidjan             Côte d'Ivoire   
1642          Montréal                    Canada   
1643          Oreamuno                Costa Rica   

                                     Plan Name  \
0         Amsterdam: a different energy (SEAP)   
1             Amsterdam definitely sustainable   
2                        New Amsterdam climate   
3     Outspokenly sustainable-perspective 2014   
4          Structure vision for Amsterdam 2014   
...                                        ...   
1639                     

In [169]:
merged_dataset = existing_plans[~pd.isna(existing_plans['matched_cities'])].merge(
    city_list, left_on='matched_cities', right_on='city_code', how='left'
)

# Display the result
print(merged_dataset)

#write cleaned up list to file
merged_dataset.to_csv('output/merged-data.csv')

             City      Country                             Plan Name  \
0       Amsterdam  Netherlands  Amsterdam: a different energy (SEAP)   
1       Amsterdam  Netherlands  Amsterdam: a different energy (SEAP)   
2       Amsterdam  Netherlands  Amsterdam: a different energy (SEAP)   
3       Amsterdam  Netherlands  Amsterdam: a different energy (SEAP)   
4       Amsterdam  Netherlands  Amsterdam: a different energy (SEAP)   
...           ...          ...                                   ...   
237304   Montréal       Canada        climate_plan_2020_2030_vdm.pdf   
237305   Montréal       Canada        climate_plan_2020_2030_vdm.pdf   
237306   Montréal       Canada        climate_plan_2020_2030_vdm.pdf   
237307   Montréal       Canada        climate_plan_2020_2030_vdm.pdf   
237308   Montréal       Canada        climate_plan_2020_2030_vdm.pdf   

       Plan Name (English translation) Plan Type Plan Year  \
0                                  NaN      SEAP      2010   
1          

In [143]:
# import pandas as pd

# # Example DataFrames
# data_existing_plans = {'iso_code': ['CAN', 'CAN'],
#                        'city': [ 'Toronto', 'Vancouver']}
# data_city_list = {'iso_code': ['USA', 'USA', 'CAN', 'CAN'],
#                   'city_code': ['NYC', 'LA', 'TOR', 'VAN'],
#                   'city': ['New York City', 'Los Angeles', 'Toronto', 'Vancouver']}

# existing_plans = pd.DataFrame(data_existing_plans)
# city_list = pd.DataFrame(data_city_list)

# # Function to find matching city code
# def find_matching_city(row, city_list_group):
#     city_list_group_lower = city_list_group['city'].str.lower()
#     row_city_lower = row['city'].lower()

#     # Check if the city names match or if one is contained in the other
#     matches = city_list_group_lower.apply(lambda city: city.lower() in row_city_lower or row_city_lower in city.lower())
#     matching_city_codes = city_list_group.loc[matches, 'city_code'].tolist()

#     if matching_city_codes:
#         return ', '.join(matching_city_codes)
#     else:
#         return None

# # Apply the custom function to find matching city codes within each group
# existing_plans['matched_cities'] = existing_plans.apply(
#     lambda row: find_matching_city(row, city_list.groupby('iso_code').get_group(row['iso_code'])),
#     axis=1
# )

# # Display the result
# print(existing_plans)


  iso_code       city matched_cities
0      CAN    Toronto            TOR
1      CAN  Vancouver            VAN


In [79]:
###OLD 
#Merge existing list of plans and desired city lists
existing_plans_cleaned = pd.read_csv('data/existing-plans-manually-cleaned.csv')

#few more cleaning steps
existing_plans_cleaned['Country'] = existing_plans_cleaned['Country'].replace('US', 'United States of America')

#Group by country and then compare city names for better probability of matching


#group by country
grouped_cities = city_list.groupby(['Country or area'])
grouped_city_plans = existing_plans_cleaned.groupby('Country')

def match_cities(cities_group, plans_group):
    #give similarity score for all city
    for plan_city in plans_group['City']:
        for city in cities_group:
            print(city, plan_city)
        # print(row['city'], city)
            if fuzz.ratio(city, plan_city) > 80:
                return city
    return None


for country, city_group in grouped_cities: 
    # print(city_group)
    try:
        #Get cities and info for that country
        # print(country[0])
        city_plans_group = grouped_city_plans.get_group(country[0])
        print(city_plans_group)
        #go through cities in the group, does it match any of the cities in the plans? 
        #if so, add the city name to the column 'matched city', 'code', 'population' to city plans?
        # city_group.apply(match_cities, plans_group=city_plans_group)
        # print(city_plans_group['matched_city'])
    except KeyError:
        pass
# print(city_plans_group[['City', 'matched_city']])
#for each country compare name of city in city_list with name of city in exisiting list, look through all 
# city_list_grouped_country['Country of area']





      Unnamed: 0               Authority    City  Country Name of the plan  \
1420        1422  Municipality of Tirana  Tirana  Albania              NaN   

     English Translation (if available) Plan Type  Year  \
1420                                NaN       NaN  2014   

                    Plan Name or Link (need to extract)  \
1420  https://mycovenant.eumayors.eu/docs/seap/13628...   

                                              Plan Link  Municipal Link  \
1420  https://mycovenant.eumayors.eu/docs/seap/13628...             NaN   

                        Dataset Source  Downloaded  
1420  CDP 2023 Adaptation Action Plans         NaN  
      Unnamed: 0                                   Authority  \
34            36                                         NaN   
467          469                             General Lavalle   
480          482                 Municipalidad de Avellaneda   
481          483                             General Ramirez   
487          489            

In [74]:
import pandas as pd
from fuzzywuzzy import fuzz

# Example DataFrames
data1 = {'City': ['New York', 'Los Angeles', 'Chicago', 'Houston']}
data2 = {'City': ['NYC', 'Los Angeles', 'Chicago', 'H-Town', 'Miami']}

df1 = pd.DataFrame(data1)
df2 = pd.DataFrame(data2)

# Function to calculate similarity score using fuzzywuzzy
def calculate_similarity(row):
    return fuzz.ratio(row['City_x'], row['City_y'])

# Merge the DataFrames based on cross join (Cartesian product)
merged_df = pd.merge(df1.assign(key=1), df2.assign(key=1), on='key').drop('key', axis=1)

# Calculate similarity score for each pair of cities
merged_df['Similarity'] = merged_df.apply(calculate_similarity, axis=1)

# Find the maximum similarity score for each city in df1
max_similarity_df1 = merged_df.groupby('City_x')['Similarity'].max().reset_index()

# Merge the max_similarity_df1 with df1 to get the final result
result_df = pd.merge(df1, max_similarity_df1, left_on='City', right_on='City_x', how='left')
result_df = result_df.rename(columns={'City_x': 'City_in_df2', 'Similarity': 'SimilarityScore'})

print(result_df)

          City  City_in_df2  SimilarityScore
0     New York     New York               36
1  Los Angeles  Los Angeles              100
2      Chicago      Chicago              100
3      Houston      Houston               46


In [18]:
from fuzzywuzzy import fuzz

string1 = "Municipalidad de Santiago"
string2 = "Santiago"

similarity_score = fuzz.ratio(string1, string2)
print(f"Similarity Score: {similarity_score}")


Similarity Score: 48


/Users/modekurt/Library/Python/3.9/lib/python/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [27]:
import pandas as pd

def compare_col2_values_for_subset(df1, df2, col1, col2, subset_column='Country'):
    """
    Compare values in col1 for a subset of rows in df1 with values in col2 for a subset of rows in df2,
    where the subset is based on the values in a specified column.

    Parameters:
    - df1: DataFrame
    - df2: DataFrame
    - col1: str, name of the column in df1
    - col2: str, name of the column in df2
    - subset_column: str, name of the column used for subset selection (default: 'Country')

    Returns:
    - DataFrame with matching rows and additional columns indicating the comparison results
    """
    # Merge the DataFrames based on the subset_column
    merged_df = pd.merge(df1, df2, how='inner', on=subset_column, suffixes=('_df1', '_df2'))

    # Compare values in col1 for the subset of rows in df1 with values in col2 for the subset of rows in df2
    merged_df['IsContained'] = merged_df.apply(lambda row: row[col1 + '_df1'] in row[col2 + '_df2'], axis=1)

    return merged_df

# Example usage:
data1 = {'Text': ['apple', 'orange', 'banana'],
         'Country': ['US', 'Canada', 'US']}
df1 = pd.DataFrame(data1)

data2 = {'Description': ['apple pie', 'orange juice', 'grapefruit'],
         'Country': ['US', 'Canada', 'US']}
df2 = pd.DataFrame(data2)

result_df = compare_col2_values_for_subset(df1, df2, 'Text', 'Description', subset_column='Country')
print(result_df)


KeyError: 'Text_df1'